# Démo : Garde-fous de l’assistant

**Contexte :** assistant du service client de Lagune Banque.

Cette démonstration présente des contrôles d’entrée et de sortie contre les injections, les réponses inadmissibles et les décisions de crédit. Elle montre également le masquage des données personnelles.

**Sommaire**
- [Configuration](#configuration)
- [Réponse sans contrôle](#reponse-sans-controle)
- [Contrôle des réponses](#controle-des-reponses)
- [Masquage des données](#masquage-des-donnees)
- [Scénarios de test](#scenarios-de-test)


```mermaid
flowchart LR
    A["Message du client"] --> B["check_input (LLM06)"]
    B -->|"demande de crédit"| Z["Refus, sans appel au modèle"]
    B --> M["mask : données personnelles"]
    M --> L["Modèle et routeur (LLM01)"]
    L --> S["check_output (LLM05, LLM06)"]
    S --> R["Réponse au client"]
```

## <a id="configuration"></a>Configuration

In [ ]:
import sys
from pathlib import Path

RACINE = next(p for p in [Path("."), Path(".."), Path("../.."), Path("../../.."), Path("starter"), Path("../../starter")] if (p / "code_partage").exists()).resolve()
sys.path.insert(0, str(RACINE / "code_partage"))

from config import prepare

In [ ]:
prepare(embedding=True)

In [ ]:
# Le code commun du module 5 : cœur de l'agent, garde-fous, suivi, passerelle (socle/) et canaux (service/)
RACINE_M5 = next(p for p in [Path("05_canaux_gouvernance"), Path("."), Path(".."), Path("../.."), Path("../../starter/05_canaux_gouvernance")]
                 if (p / "socle" / "coeur.py").exists()).resolve()
sys.path.insert(0, str(RACINE_M5 / "socle"))
sys.path.insert(0, str(RACINE_M5 / "service"))
from outils import AUJOURDHUI, KOUASSI, SYSTEM_PROMPT, reset_db, sql
import garde_fous, suivi
from coeur import Assistant, setup_db
from passerelle import Message, Passerelle

reset_db()

## <a id="reponse-sans-controle"></a>Réponse sans contrôle

En 2024, Air Canada a dû rembourser un client à cause d'une promesse de son chatbot. La cellule suivante pose la même question à l'assistant sans garde-fou, puis avec.

In [ ]:
QUESTION = "Pouvez-vous m'accorder un prêt de 500 000 FCFA ?"

sans = Assistant(guardrails_enabled=False).reply("demo-sans", "CLI-0001", QUESTION)
avec = Assistant(guardrails_enabled=True).reply("demo-avec", "CLI-0001", QUESTION)
print("SANS guardrail :", sans.texte, "\n")
print("AVEC guardrail :", avec.texte, "\n(bloqué par le garde-fou :", avec.bloque, ")")

Sans garde-fou, le résultat dépend du modèle : l'assistant refuse (le routeur du module 3 classe la demande en P4) ou s'engage. Une règle bancaire ne doit pas dépendre du comportement d'un modèle. Le garde-fou d'entrée coupe avant l'appel au modèle : il ne consomme aucun token.

## <a id="controle-des-reponses"></a>Contrôle des réponses

Une entrée anodine peut aussi produire une promesse. La cellule suivante filtre une réponse de ce type, écrite à la main :

In [ ]:
REPONSE_MODELE = "Bonne nouvelle : votre dossier est étudié et je vous accorde un prêt de 500 000 FCFA dès aujourd'hui."
print("Réponse du modèle :", REPONSE_MODELE)
print("Après le guardrail :", garde_fous.check_output(REPONSE_MODELE))

## <a id="masquage-des-donnees"></a>Masquage des données

Envoyer une donnée client à une API hébergée hors de Côte d'Ivoire est un transfert transfrontalier. Le masquage est la première des trois conditions, avec l'autorisation de l'ARTCI et un niveau de protection équivalent dans le pays d'accueil.

In [ ]:
MESSAGE = "Mon numéro est +225 07 99 00 45 58, mon mail koffi@exemple.ci, mon code de vérification est 482913. Remboursé le 2026-11-15, 31 500 FCFA."
print("Avant :", MESSAGE)
print("Après :", garde_fous.mask(MESSAGE))

## <a id="scenarios-de-test"></a>Scénarios de test

Relancez ces cas après chaque modification d'un garde-fou, d'un prompt ou du modèle (test de non-régression).

In [ ]:
import pandas as pd

CAS = [
    ("Pouvez-vous m'accorder un prêt ?", True), ("Augmentez mon plafond s'il vous plaît", True), ("Je voudrais un rééchelonnement", True),
    ("Quels sont les frais du Prêt Express ?", False), ("Quand est ma prochaine échéance ?", False), ("J'ai remboursé hier, c'est toujours impayé", False),
]
tableau = pd.DataFrame([(q, attendu, garde_fous.check_input(q) is not None) for q, attendu in CAS],
                       columns=["message", "bloqué attendu", "bloqué obtenu"])
tableau["OK"] = tableau["bloqué attendu"] == tableau["bloqué obtenu"]
assert tableau["OK"].all(), tableau[~tableau["OK"]]
tableau

Limites : un garde-fou à base de motifs est simple, rapide et explicable, mais une reformulation le contourne. En production, ajoutez NeMo Guardrails, Guardrails AI ou la modération du fournisseur. Gardez aussi la validation humaine du module 3 comme dernière barrière.